# Panel 1: Reglas de Asociación (Apriori)
Este cuaderno aplica el algoritmo **Apriori** sobre el detalle de ventas para identificar qué productos se compran conjuntamente con frecuencia en el bazar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_theme(style='whitegrid')

### 1. Carga y Limpieza de Datos

In [ ]:
# Cargar detalle de ventas
df_detalle = pd.read_csv('datasets/detalle-ventas.csv', sep=';', encoding='utf-8-sig', skiprows=1)
df_detalle = df_detalle.loc[:, ~df_detalle.columns.str.contains('^Unnamed')]
df_detalle = df_detalle.dropna(subset=['ID_Venta', 'Descripcion'])
print(f"Detalle cargado: {df_detalle.shape[0]} registros")

### 2. Preparación de la Matriz de Transacciones (One-Hot Encoding)
Creamos la matriz cruzada de tickets vs. descripciones de productos.

In [ ]:
# Pivotar a formato transaccional (1 si el producto está en la compra, 0 de lo contrario)
basket = (df_detalle.groupby(['ID_Venta', 'Descripcion'])['Cantidad']
          .sum().unstack().reset_index().fillna(0)
          .set_index('ID_Venta'))

basket_sets = basket.map(lambda x: 1 if x > 0 else 0)
print(f"Matriz de transacciones: {basket_sets.shape[0]} transacciones y {basket_sets.shape[1]} ítems.")

### 3. Algoritmo Apriori y Generación de Reglas
Ajustamos un soporte mínimo y confianza mínimos para extraer las reglas.

In [ ]:
MIN_SUPPORT = 0.015
MIN_CONFIDENCE = 0.20

# Extraer itemsets frecuentes
frequent_itemsets = apriori(basket_sets, min_support=MIN_SUPPORT, use_colnames=True)
print(f"Cantidad de itemsets frecuentes encontrados: {len(frequent_itemsets)}")

# Generar reglas
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules[rules['confidence'] >= MIN_CONFIDENCE]
print(f"Cantidad de reglas generadas: {len(rules)}")

### 4. Selección e Interpretación de Combos Promocionales
Ordenamos las reglas por el indicador `lift` para extraer los combos más potentes.

In [ ]:
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ", ".join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ", ".join(list(x)))

top_combos = rules.sort_values(by='lift', ascending=False).head(5)
for idx, row in top_combos.iterrows():
    print(f"Combo: {row['antecedents_str']} => {row['consequents_str']}")
    print(f" - Soporte (Frecuencia relativa): {row['support']:.2%}")
    print(f" - Confianza (Fuerza condicional): {row['confidence']:.2%}")
    print(f" - Lift (Factor de asociación): {row['lift']:.2f}\n")

### 5. Visualización del Espacio de Reglas

In [ ]:
plt.figure(figsize=(10, 5))
scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='plasma', s=rules['lift']*50, alpha=0.8)
plt.colorbar(scatter, label='Lift')
plt.title("Mapa de Reglas de Asociación (Soporte vs Confianza)")
plt.xlabel("Soporte")
plt.ylabel("Confianza")
plt.grid(True)
plt.show()